# Week 03 — Function 03

## Table of contents

1. [Overview](#overview)
2. [Objectives](#objectives)
3. [Evidence provenance](#evidence-provenance)
4. [Environment and setup](#environment-setup)
5. [Data validation](#data-validation)
6. [Descriptive EDA](#descriptive-eda)
7. [Visual EDA](#visual-eda)
8. [Model and acquisition](#model-acquisition)
9. [Week 3 proposal](#week-3-proposal)
10. [Reproducibility checks](#reproducibility-checks)
11. [Conclusions and next steps](#conclusions-next-steps)

<a id="overview"></a>
## 1. Overview

This focused review mirrors the canonical Week 3 methodology for Function 03, with Weeks 1–2 observed and Week 3 proposed only.

<a id="objectives"></a>
## 2. Objectives

Validate the 3-dimensional evidence, assess the latest returned point, and reproduce the recorded GP-UCB proposal without look-ahead.

<a id="evidence-provenance"></a>
## 3. Evidence provenance

Starter arrays come from `Week_01/Function_nn/03_Data`; exact returned pairs come from `Results/query_output_ledger.csv`. The Week 3 return is excluded because it was unknown when the proposal was selected.

<a id="environment-setup"></a>
## 4. Environment and setup

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path.cwd().resolve()
for candidate in (ROOT,*ROOT.parents):
    if (candidate/'Week_03'/'Function_03').is_dir(): ROOT=candidate; break
else: raise FileNotFoundError('Could not locate repository root')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from Code.historical_function_review import analyse_historical_function

<a id="data-validation"></a>
## 5. Data validation

In [2]:
observations, summary, proposal, diagnostic_figure = analyse_historical_function(3, 3, ROOT)
input_columns=[f'x{i}' for i in range(1,4)]
inputs=observations[input_columns].to_numpy(float)
outputs=observations['objective'].to_numpy(float)
assert inputs.shape==(17,3) and outputs.shape==(17,)
assert np.isfinite(inputs).all() and np.isfinite(outputs).all()
assert np.all((inputs>=0)&(inputs<=1))
observations

,query,evidence,x1,x2,x3,objective
0,1,starter,0.171525,0.343917,0.248737,-0.112122
1,2,starter,0.242114,0.644074,0.272433,-0.087963
2,3,starter,0.534906,0.398501,0.173389,-0.111415
3,4,starter,0.492581,0.611593,0.340176,-0.034835
4,5,starter,0.134622,0.219917,0.458206,-0.048008
5,6,starter,0.345523,0.941360,0.269363,-0.110621
6,7,starter,0.151837,0.439991,0.990882,-0.398926
7,8,starter,0.645503,0.397143,0.919771,-0.113869
8,9,starter,0.746912,0.284196,0.226300,-0.131461
9,10,starter,0.170477,0.697032,0.149169,-0.094190


<a id="descriptive-eda"></a>
## 6. Descriptive EDA

All comparisons are descriptive and within-function; no causal, global-optimum, or cross-function ranking claim is made.

In [3]:
pd.Series({k:v for k,v in summary.items() if k!='proposal'}, name='verified evidence')

week                                                                             3
function                                                                         3
dimensions                                                                       3
starter_observations                                                            15
recorded_pairs                                                                   2
total_verified_observations                                                     17
best_query                                                                       4
best_input                       [0.49258141463713434, 0.6115931882759961, 0.34...
best_output                                                              -0.034835
latest_verified_query                                                           17
latest_verified_input                               [0.657452, 0.998464, 0.817253]
latest_verified_output                                                   -0.089875
late

<a id="visual-eda"></a>
## 7. Visual EDA

Orange markers are returned Weeks 1–2 observations; the star is the verified incumbent. The Week 3 proposal is deliberately absent.

In [4]:
display(diagnostic_figure)
plt.close(diagnostic_figure)

<Figure size 1200x450 with 3 Axes>

<a id="model-acquisition"></a>
## 8. Model and acquisition

Method: **GP-UCB**. This adaptive policy is a heuristic chosen from the evidence available at the decision boundary; it is not a statistically controlled acquisition comparison.

<a id="week-3-proposal"></a>
## 9. Week 3 proposal

Proposed only: `[0.670026, 0.057881, 0.658241]`. Decision record: Chosen using evidence through Week 2, before the Week 3 return.

<a id="reproducibility-checks"></a>
## 10. Reproducibility checks

In [5]:
candidate=np.asarray(proposal['query'],dtype=float)
assert proposal['status']=='proposed_only'
assert candidate.shape==(3,) and np.all((candidate>=0)&(candidate<=0.999999))
duplicate=bool(np.any(np.all(np.isclose(inputs,candidate,rtol=0,atol=5e-7),axis=1)))
assert duplicate==proposal['duplicates_observed_evidence']
assert summary['recorded_pairs']==2
portal='-'.join(f'{value:.6f}' for value in candidate)
assert all(len(part.split('.')[-1])==6 for part in portal.split('-'))
print('Function 03 Week 3 checks passed:', portal, 'duplicate:', duplicate)

Function 03 Week 3 checks passed: 0.670026-0.057881-0.658241 duplicate: False


<a id="conclusions-next-steps"></a>
## 11. Conclusions and next steps

The evidence boundary is locked at 17 verified observations. The Week 3 proposal remains unobserved until its authoritative return is appended at the next checkpoint.